# Notebook 01 — Tracking Pipeline

Full pipeline: RF-DETR detection -> OC-SORT tracking -> vial assignment -> overlays.

## Stages
1. Setup & configuration
2. (Optional) Background subtraction
3. Draw vial ROIs
4. RF-DETR + OC-SORT tracking -> ocsort_tracks.csv
5. Vial assignment + ordered IDs -> ordered_tracks.csv
6. Diagnostics
7. Overlay video rendering


In [ ]:
import sys
sys.path.insert(0, '..')

import json
import os
import re
import cv2
import yaml
import pandas as pd
from pathlib import Path
from IPython.display import Video

from src.preprocessing import preprocess_bgsub_gui
from src.metrics import run_diagnostics
from src.tracking import export_tracks_xy_tuple_csv_one_config
from src.stitching import wide_to_long
from src.roi import draw_and_save_vial_rois, assign_vials_and_ordered_ids
from src.visualization import render_vial_overlay_video, render_raw_overlay_video, render_detections_video
from utils import save_run_params, load_config

## 1 - Configuration

Set your paths and Roboflow credentials here.

In [ ]:
# ---- EDIT THESE ----
RAW_VIDEO = r"../2024-02-05_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m/13 DPE/002/2024-02-12_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_13d_002-converted.mp4"

with open("../creds_config.yaml") as f:
    creds_config = yaml.safe_load(f)
API_KEY = creds_config["API_KEY"]

cfg = load_config("../config.yaml")
MODEL_ID = creds_config.get("MODEL_ID") or cfg.roboflow.model_id

# Auto-increment output directory.
_m = re.search(r'(\d+)\s+DPE[/\\](\d+)', RAW_VIDEO)
short_name = f"{_m.group(1)}DPE_n{_m.group(2).zfill(3)}" if _m else Path(RAW_VIDEO).stem[:20]

_outputs_root = Path("../outputs")
_outputs_root.mkdir(parents=True, exist_ok=True)
_existing = [d for d in _outputs_root.iterdir() if d.is_dir() and d.name.startswith("run_")]
_next_n = max((int(d.name.split("_")[1]) for d in _existing if d.name.split("_")[1].isdigit()), default=0) + 1
_dir_name = f"run_{_next_n}_{_m.group(1)}DPE_n{_m.group(2).zfill(3)}" if _m else f"run_{_next_n}"
OUTPUT_PATH = str(_outputs_root / _dir_name)
os.makedirs(OUTPUT_PATH, exist_ok=True)

import shutil
_dest_video = os.path.join(OUTPUT_PATH, Path(RAW_VIDEO).name)
if not os.path.exists(_dest_video):
    try:
        os.link(RAW_VIDEO, _dest_video)
    except OSError:
        shutil.copy2(RAW_VIDEO, _dest_video)
PATH_TO_VID = RAW_VIDEO

print("Output dir:", OUTPUT_PATH)
print("Short name:", short_name)
print(f"inference_api_url={cfg.roboflow.inference_api_url}")
print(f"asso_func={cfg.tracker.asso_func}, detection_confidence_rfdetr={cfg.tracker.detection_confidence_rfdetr}")
print(f"aspect_weight={cfg.tracker.aspect_weight}, behavioral_weight={cfg.tracker.behavioral_weight}")
print(f"jump_factor={cfg.tracker.jump_factor}, jump_iou_threshold={cfg.tracker.jump_iou_threshold}, jump_inertia={cfg.tracker.jump_inertia}")

_cap = cv2.VideoCapture(RAW_VIDEO)
fps = float(_cap.get(cv2.CAP_PROP_FPS) or cfg.video.fallback_fps)
save_run_params(OUTPUT_PATH, "config", {
    "video": RAW_VIDEO, "output_dir": OUTPUT_PATH, "short_name": short_name,
    "video_fps": fps,
    "video_width": int(_cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
    "video_height": int(_cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    "video_frames": int(_cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    "tracker": dict(cfg.tracker),
    "preprocessing": dict(cfg.preprocessing),
})
_cap.release()

## 2 - (Optional) Background subtraction + temporal trim

Opens a GUI to:
1. Draw a **crop ROI** (spatial - pixels outside it are removed).
2. Pick a **start/end frame range** (temporal - frames outside `[start, end)` are *discarded*; only frames inside are kept).

The output `_pp.mp4` is shorter than the source: spatially cropped, temporally trimmed, and with the **85th-percentile** background subtracted. All downstream steps (tracking, stitching, overlays) run on this trimmed clip.

In [ ]:
ROI_LIBRARY = Path("../roi_library.json")
_video_key = Path(RAW_VIDEO).stem

# Load existing library (or start fresh)
if ROI_LIBRARY.exists():
    with open(ROI_LIBRARY) as f:
        _library = json.load(f)
else:
    _library = {}

_crop_params = None
RAW_CROPPED_VIDEO = None
_use_saved_roi = cfg.roi.use_saved_roi
preprocess = True  # set to False to skip bg subtraction entirely

if preprocess:
    pp_out = os.path.join(OUTPUT_PATH, Path(RAW_VIDEO).stem + "_pp.mp4")
    raw_cropped_out = os.path.join(OUTPUT_PATH, Path(RAW_VIDEO).stem + "_raw_cropped.mp4")
    _stored_crop = _library.get(_video_key, {}).get("preprocessing") if _use_saved_roi else None

    if _use_saved_roi and _stored_crop is not None:
        print(f"Found stored preprocessing params for: {_video_key}")
    else:
        if not _use_saved_roi:
            print("use_saved_roi=False - opening preprocessing GUI...")
        else:
            print(f"No stored preprocessing params for: {_video_key} - opening GUI...")

    pp_path, _crop_params = preprocess_bgsub_gui(
        video_path=RAW_VIDEO,
        out_mp4=pp_out,
        out_raw_mp4=raw_cropped_out,
        gain=cfg.preprocessing.bg_gain,
        white_level=cfg.preprocessing.bg_white_level,
        bg_sample_stride=cfg.preprocessing.bg_sample_stride,
        bg_percentile=cfg.preprocessing.bg_percentile,
        crop_params=_stored_crop if _use_saved_roi else None,
    )
    PATH_TO_VID = Path(pp_path)
    RAW_CROPPED_VIDEO = Path(raw_cropped_out)

    # Save crop params + full video path to library
    if _video_key not in _library:
        _library[_video_key] = {}
    _library[_video_key]["preprocessing"] = _crop_params
    _library[_video_key]["video_path"] = RAW_VIDEO
    ROI_LIBRARY.parent.mkdir(parents=True, exist_ok=True)
    with open(ROI_LIBRARY, "w") as f:
        json.dump(_library, f, indent=2)
    print("Preprocessing params saved to library.")

    # Save crop_roi.json to this run folder (allows skipping GUI on re-runs)
    with open(os.path.join(OUTPUT_PATH, "crop_roi.json"), "w") as _f:
        json.dump(_crop_params, _f, indent=2)

save_run_params(OUTPUT_PATH, "preprocessing",
                {"video_pp": str(PATH_TO_VID), "video_raw_cropped": str(RAW_CROPPED_VIDEO) if RAW_CROPPED_VIDEO is not None else None, "crop_params": _crop_params})


## 3 - Draw vial ROIs

Opens an OpenCV GUI on frame 0: drag rectangles around each vial.
Press **q** when all 6 ROIs are drawn. Saved to `vial_rois.json`.

This is a one-time step - reuse the JSON for the same experimental setup.

In [ ]:
ROI_JSON = os.path.join(OUTPUT_PATH, "vial_rois.json")
_use_saved_roi = cfg.roi.use_saved_roi
_stored_vials = _library.get(_video_key, {}).get("vial_rois")

if _use_saved_roi and _stored_vials is not None:
    print(f"Found stored vial ROIs for: {_video_key}")
    _vials = {k: tuple(v) for k, v in _stored_vials.items()}
    with open(ROI_JSON, "w") as f:
        json.dump({k: list(v) for k, v in _vials.items()}, f, indent=2)
    print(f"Loaded {len(_vials)} vials from library.")
else:
    if not _use_saved_roi:
        print("use_saved_roi=False - opening GUI...")
    else:
        print(f"No stored vial ROIs for: {_video_key} - opening GUI...")
    _vials = draw_and_save_vial_rois(video_path=RAW_VIDEO, roi_json_path=ROI_JSON)

    # Save to library
    if _video_key not in _library:
        _library[_video_key] = {}
    _library[_video_key]["vial_rois"] = {k: list(v) for k, v in _vials.items()}
    ROI_LIBRARY.parent.mkdir(parents=True, exist_ok=True)
    with open(ROI_LIBRARY, "w") as f:
        json.dump(_library, f, indent=2)
    print("Vial ROIs saved to library.")

save_run_params(OUTPUT_PATH, "roi", {k: list(v) for k, v in _vials.items()})

## 4 - RF-DETR + OC-SORT tracking

Runs the detector + tracker on every frame and writes a wide CSV.
This is the most time-consuming step. 

In [ ]:
OCSORT_CSV  = os.path.join(OUTPUT_PATH, "ocsort_tracks.csv")
DET_LOG_CSV = os.path.join(OUTPUT_PATH, "detections_raw.csv")

#  Detection cache 
# Set CACHED_DETS to a previous run's detections_raw.csv to skip RF-DETR.
# Leave as None to run inference and save fresh detections to DET_LOG_CSV.
CACHED_DETS = None

_det_source = CACHED_DETS if (CACHED_DETS and os.path.exists(CACHED_DETS)) else DET_LOG_CSV
if CACHED_DETS and os.path.exists(CACHED_DETS):
    print(f"Using cached detections: {CACHED_DETS}")
else:
    print("No cache found - running RF-DETR inference")

t = cfg.tracker
df_wide, tracker, df_relinked = export_tracks_xy_tuple_csv_one_config(
    video_path=str(PATH_TO_VID),
    output_csv=OCSORT_CSV,
    api_key=API_KEY,
    model_id=MODEL_ID,
    inference_api_url=cfg.roboflow.inference_api_url,
    detection_confidence_rfdetr=t.detection_confidence_rfdetr,
    confidence=t.confidence,
    lost_track_buffer=t.lost_track_buffer,
    minimum_matching_threshold=t.minimum_matching_threshold,
    minimum_consecutive_frames=t.minimum_consecutive_frames,
    asso_func=t.asso_func,
    brownian_pos_noise=t.brownian_pos_noise,
    aspect_weight=t.aspect_weight,
    behavioral_weight=t.behavioral_weight,
    jump_factor=t.jump_factor,
    jump_iou_threshold=t.jump_iou_threshold,
    jump_inertia=t.jump_inertia,
    det_log_csv=_det_source,
    vial_rois=_vials,
    max_frames=None,
    relinked_csv=os.path.join(OUTPUT_PATH, "tracks_relinked.csv"),
)

print(df_wide.shape)
save_run_params(OUTPUT_PATH, "tracker_output", {
    "ocsort_csv": OCSORT_CSV, "frames": int(df_wide.shape[0]), "track_count": int(df_wide.shape[1] - 1),
})
df_wide.head()

with open(os.path.join(OUTPUT_PATH, "tracker_log.json"), "w") as _f:
    json.dump({
        "detection_log":     tracker.detection_log,
        "suppressed_tracks": tracker.suppressed_tracks,
        "min_hits":          tracker.min_hits,
        "max_age":           tracker.max_age,
    }, _f)

render_detections_video(
    video_path=str(PATH_TO_VID),
    det_log_csv=_det_source,
    out_mp4=os.path.join(OUTPUT_PATH, f"{short_name}_detections_RF-DETR.mp4"),
)

In [ ]:
# Quick mid-pipeline check: are detections reaching the tracker?
# No vial assignment yet, so no per-vial report is saved here.
run_diagnostics(
    tracker     = tracker,
    df_wide     = df_wide,
    df_relinked = df_relinked,
    n_expected  = cfg.pipeline.expected_per_vial * len(_vials),
    fps         = fps,
    config      = cfg,
)

## 5 — Vial assignment + ordered IDs

`wide_to_long` melts the wide OC-SORT CSV into long format. Each detection is
then assigned to a vial via the ROI JSON, and `ordered_id` is a left-to-right
sequential ID within each vial.

In [ ]:
LONG_CSV    = os.path.join(OUTPUT_PATH, "ocsort_tracks_long.csv")
ORDERED_CSV = os.path.join(OUTPUT_PATH, "ordered_tracks.csv")

with open(ROI_JSON) as f:
    vial_rois = {k: tuple(v) for k, v in json.load(f).items()}

long_df = wide_to_long(pd.read_csv(OCSORT_CSV), out_csv=LONG_CSV)

df_ordered = assign_vials_and_ordered_ids(
    ocsort_csv=LONG_CSV,
    roi_json=ROI_JSON,
    out_csv=ORDERED_CSV,
    fps=fps,
)

print(f"Track IDs: {long_df['orig_id'].nunique()}  ->  ordered IDs: {df_ordered['ordered_id'].nunique()}")
save_run_params(OUTPUT_PATH, "ordered", {
    "csv": ORDERED_CSV,
    "rows": int(df_ordered.shape[0]),
    "track_count": int(df_ordered["ordered_id"].nunique()),
})
df_ordered.head()

## 6 — Diagnostics

Per-vial sanity checks (track counts vs. expected, suppressed tracks,
re-link swaps). Writes a diagnostics report into the run folder.

In [ ]:
df_wide = pd.read_csv(OCSORT_CSV)

run_diagnostics(
    tracker    = tracker,
    df_wide    = df_wide,
    df_ordered = df_ordered,
    df_relinked= df_relinked if "df_relinked" in dir() else None,
    n_expected = cfg.pipeline.expected_per_vial * len(vial_rois),
    fps        = fps,
    vial_rois  = vial_rois,
    config     = cfg,
    output_dir = OUTPUT_PATH,
)

## 7 - Overlay video

Renders each fly as a coloured dot on the original video.

In [ ]:
RAW_OVERLAY_MP4 = os.path.join(OUTPUT_PATH, f"{short_name}_overlay_raw_ocsort.mp4")
OVERLAY_MP4     = os.path.join(OUTPUT_PATH, f"{short_name}_overlay_ordered.mp4")

# Pick the overlay substrate from config.yaml:visualization.overlay_source.
# Kept separate from PATH_TO_VID, which is the tracker input (_pp after Stage 2).
# In raw_cropped mode, prefer the saved cropped-raw clip when preprocessing ran.
# Fall back to RAW_VIDEO when no cropped-raw artifact exists.
_overlay_mode = cfg.visualization.overlay_source.lower()
if _overlay_mode == "raw_cropped" and RAW_CROPPED_VIDEO is not None:
    OVERLAY_VIDEO = str(RAW_CROPPED_VIDEO)
elif _overlay_mode == "raw_cropped":
    OVERLAY_VIDEO = RAW_VIDEO
else:
    OVERLAY_VIDEO = str(PATH_TO_VID)
print(f"overlay_source={_overlay_mode}  ->  substrate: {OVERLAY_VIDEO}")

render_raw_overlay_video(
    video_path=OVERLAY_VIDEO,
    csv_path=LONG_CSV,
    out_mp4=RAW_OVERLAY_MP4,
    vial_rois=vial_rois,
    det_log_csv=DET_LOG_CSV,
)

render_vial_overlay_video(
    video_path=OVERLAY_VIDEO,
    csv_path=ORDERED_CSV,
    out_mp4=OVERLAY_MP4,
    vial_rois=vial_rois,
    det_log_csv=DET_LOG_CSV,
)

save_run_params(OUTPUT_PATH, "outputs", {"raw_overlay": RAW_OVERLAY_MP4, "ordered_overlay": OVERLAY_MP4})
Video(RAW_OVERLAY_MP4, width=800)